# Review-2 Demonstration

**Adaptive Emotional Digital Twin Using Continual Knowledge Hypergraph Reasoning**

> ## ⚠️ EVERY RESULT IN THIS NOTEBOOK IS **SYNTHETIC**
>
> No dataset file has been opened by this project. Nothing here is evidence
> about humans. Every figure and table carries a visible `SYNTHETIC` badge.

The same nine stages the terminal demo prints, with the figures inline.
The one-command equivalent is:

```bash
python scripts/run_demo.py --dataset synthetic
```

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd
from aedt.config import load_config
from aedt.logging_setup import setup_logging
from aedt.pipeline import run_pipeline

setup_logging("INFO")
cfg = load_config("simulation")
print("true rho used by the generator:", cfg.get("simulation.true_rho"),
      "| seed:", cfg.seed, "| deviations from frozen spec:",
      cfg.deviations_from_frozen())

## Stages 1-5 — data, reports, sensor, context, twin

`run_pipeline` enforces the frozen execution order: ingest → preprocess →
epochs → features/context → audit → **[9b] association** → eligibility →
**placebo** → primary → uncertainty → twin update.

In [ ]:
res = run_pipeline("synthetic", config=cfg, n_resamples=999)
print("DATA STATUS:", res.data_status.value)
print("participants:", res.frame['pid'].nunique(), "| observations:", len(res.frame))
print("eligible:", res.n_eligible, "/", len(res.eligibility))
print("[9b] median |beta|:", round(res.association.median_abs_beta, 3),
      "| weak:", res.association.weak)

In [ ]:
# Stage 2 — category usage. Most days sit at one end of the scale.
from aedt.viz import category_usage_plot
category_usage_plot(res.category_usage, res.n_categories,
                    data_status=res.data_status)

In [ ]:
# Stage 5 — the Personal Digital Twin. It models the MEASURING INSTRUMENT.
from aedt.demo_artefacts import representative_participant
pid = representative_participant(res)
twin = res.twins[pid]
print(twin)
print("audit flags:", twin.state.audit_flags)
print("knowledge nodes:", len(twin.knowledge.nodes))
twin.knowledge.to_frame().tail(4)[['kind', 'valid_from', 'provenance']]

## Stage 6 — ⭐ THE MONEY SHOT

Epoch 1 vs epoch 2 ordinal curves on one axis.

> *"If the curve got flatter, the same behaviour now earns a different number."*

The featured participant is selected by a rule fixed in advance — the one
whose own ρ\* is **closest to the cohort ρ\***. Representative, never
cherry-picked.

In [ ]:
from aedt.demo_artefacts import two_curve_figure
import tempfile, pathlib
from IPython.display import Image
p = two_curve_figure(res, pid,
                     pathlib.Path(tempfile.mkdtemp()) / "two_curve.png",
                     selection_rule=f"Participant {pid}: own rho* closest to the cohort rho*.")
Image(str(p))

## Stage 7 — the audit, which runs BEFORE the result

The placebo splits **epoch 1** into two contiguous halves. No response shift
can exist between them, so the estimator must not reject. It **gates** the
primary.

In [ ]:
from aedt.viz import placebo_plot
print(res.placebo.verdict)
placebo_plot(res.placebo, res.primary)

## Stage 8 — the result

**ρ\* is the identified estimand.** ρ itself is not point-identified and the
additive component is provably not identifiable.

In [ ]:
from aedt.reporting.tables import estimator_table
estimator_table(res.primary).T

In [ ]:
from aedt.viz import forest_plot
forest_plot(res.primary)

## The hypergraph, and the honest ablation

The hypergraph is **not** the identification mechanism. It is the twin's
contextual knowledge representation and an ablation arm — and we report the
ablation result whichever way it falls.

In [ ]:
from aedt.viz import hypergraph_plot
hypergraph_plot(res.hypergraphs[pid], data_status=res.data_status)

In [ ]:
from aedt.hypergraph.ablation import run_context_ablation, ablation_verdict
ctx = [c for c in cfg.get('context.features', []) if c in res.frame.columns]
tab = run_context_ablation(res.frame, res.sensor, res.n_categories,
                           ctx_cols=ctx, true_rho=cfg.get('simulation.true_rho'),
                           n_resamples=399)
print(ablation_verdict(tab))
tab[['representation', 'rho_star', 'ci_width', 'effect_retention',
     'placebo_rejects']]

## Stage 9 — the twin remembers, and how much to trust it

In [ ]:
pd.DataFrame(twin.state.history)[['verdict', 'rho_star', 'reasons']]

## What is real, and what is not

In [ ]:
from aedt.io import ADAPTERS, get_adapter
from aedt.reporting.tables import (dataset_audit_table, status_board,
                                   title_alignment_table)
audits = [get_adapter(n).audit(None) for n in sorted(ADAPTERS)]
dataset_audit_table(audits)[['dataset', 'role', 'data_status',
                             'local_files_available', 'eligible_for_primary']]

In [ ]:
sb = status_board()
print('completed:', (sb.completed == 'YES').sum(),
      '| in progress:', (sb.in_progress == 'YES').sum(),
      '| planned:', (sb.planned == 'YES').sum())
sb[sb.planned == 'YES'][['item', 'evidence']]

In [ ]:
# Title term -> real module -> honest status. Slide 9b.
title_alignment_table()[['title_term', 'actual_module', 'status']]

---

**Every figure and table above is stamped SYNTHETIC. No dataset file has been
opened by this project. Real-data validation is PENDING and is the single
largest gap.**